# Polars ile Büyük Veri İşleme Klavuzu

Bu rehber, Polars kütüphanesini kullanarak büyük veri setleriyle verimli çalışmaya dair örnekler içeriyor.

In [ ]:
import polars as pl
import time
import psutil
import os

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

DATA_PATH = "/kaggle/input/trendyol-e-ticaret-hackathonu-2025-kaggle/data"

def check_memory():
    process = psutil.Process(os.getpid())
    memory_mb = process.memory_info().rss / 1024 / 1024
    print(f"Mevcut bellek kullanımı: {memory_mb:.1f} MB")
    return memory_mb

print("Polars büyük veri klavuzu hazır!")
print(f"Polars versiyonu: {pl.__version__}")
check_memory()


## 1. Lazy Evaluation

Lazy evaluation, işlemlerin gerçekten ihtiyaç duyulana kadar ertelenmesini sağlar. Bu, büyük veri setlerinde:
- Bellek kullanımını azaltır
- Query optimizasyonu sağlar
- Sadece gerekli verileri yükler

In [ ]:
# 1.1 Eager vs Lazy Loading Karşılaştırması

print("=== EAGER LOADING===")
start_time = time.time()
start_memory = check_memory()

# Eager loading - tüm veriyi hemen belleğe yükler
eager_df = pl.read_parquet(f"{DATA_PATH}/user/fashion_search_log.parquet")
print(f"Yüklenen satır sayısı: {len(eager_df):,}")
print(f"Yükleme süresi: {time.time() - start_time:.2f} saniye")
eager_memory = check_memory()
print(f"Bellek artışı: {eager_memory - start_memory:.1f} MB")

print("\n=== LAZY LOADING===")
start_time = time.time()
start_memory = check_memory()

# Lazy loading - sadece query planı oluşturur
lazy_df = pl.scan_parquet(f"{DATA_PATH}/user/fashion_search_log.parquet")
print(f"Lazy frame oluşturma süresi: {time.time() - start_time:.2f} saniye")
lazy_memory = check_memory()
print(f"Bellek artışı: {lazy_memory - start_memory:.1f} MB")

print("\nLazy frame henüz veriyi yüklemedi, sadece plan oluşturdu!")


In [ ]:
# 1.2 Lazy Chain Operations (Zincirleme İşlemler)

print("=== LAZY CHAINING ===")

lazy_query = (
    pl.scan_parquet(f"{DATA_PATH}/train_sessions.parquet")
    .filter(pl.col("session_id").is_not_null())
    .group_by("user_id_hashed")
    .agg([
        pl.count().alias("total_sessions"),
        pl.col("session_id").n_unique().alias("unique_sessions")
    ])
    .filter(pl.col("total_sessions") > 1)
    .sort("total_sessions", descending=True)
    .limit(100)
)

print("Query planı oluştu. Henüz hiçbir veri işlenmedi.")
print("\nQuery planı:")
print(lazy_query.explain())


In [ ]:
# 1.3 Lazy Query Execution

print("=== LAZY QUERY EXECUTION ===")
start_time = time.time()
start_memory = check_memory()

# Query'i çalıştır - Polars tüm optimizasyonları uygular
result = lazy_query.collect()

end_time = time.time()
end_memory = check_memory()

print(f"Query çalıştırma süresi: {end_time - start_time:.2f} saniye")
print(f"Bellek kullanımı: {end_memory - start_memory:.1f} MB")
print(f"Sonuç satır sayısı: {len(result)}")
print("\nİlk 5 sonuç:")
print(result.head())


## 2. Chunking

Büyük dosyaları küçük parçalar halinde işleyerek out-of-memory hatasından kaçınabilirsiniz.

In [ ]:
# 2.1 Row-based Chunking

def process_in_chunks_by_rows(file_path, chunk_size=50000):
    """
    Dosyayı belirtilen satır sayısında parçalar halinde işler
    """
    print(f"=== ROW-BASED CHUNKING (Chunk boyutu: {chunk_size:,} satır) ===")
    
    total_rows = pl.scan_parquet(file_path).select(pl.count()).collect().item()
    print(f"Toplam satır sayısı: {total_rows:,}")
    
    chunk_results = []
    processed_rows = 0
    
    for start_row in range(0, total_rows, chunk_size):
        end_row = min(start_row + chunk_size, total_rows)
        
        print(f"\nChunk işleniyor: {start_row:,} - {end_row:,}")
        start_memory = check_memory()
        
        # Bu chunk'ı yükle ve işle
        chunk = (
            pl.scan_parquet(file_path)
            .slice(start_row, chunk_size)
            .filter(pl.col("user_id_hashed").is_not_null())
            .group_by("user_id_hashed")
            .agg(pl.count().alias("session_count"))
            .collect()
        )
        
        chunk_results.append(chunk)
        processed_rows = end_row
        
        end_memory = check_memory()
        print(f"Chunk bellek kullanımı: {end_memory - start_memory:.1f} MB")
        print(f"Bu chunk'ta {len(chunk)} farklı kullanıcı bulundu")
        
        # İlk 3 chunk'tan sonra dur (demo için)
        if len(chunk_results) >= 3:
            print(f"\nDemo için ilk 3 chunk işlendi ({processed_rows:,} satır)")
            break
    
    return chunk_results

chunk_results = process_in_chunks_by_rows(f"{DATA_PATH}/train_sessions.parquet")


In [ ]:
# 2.2 Memory-based Chunking (Bellek Bazlı Parçalama)

def process_with_memory_limit(file_path, max_memory_mb=100):
    """
    Bellek kullanımını sınırlayarak chunk boyutunu dinamik olarak ayarlar
    """
    print(f"=== MEMORY-BASED CHUNKING (Max bellek: {max_memory_mb} MB) ===")
    
    # Küçük bir chunk ile başla ve bellek kullanımını ölç
    test_chunk_size = 10000
    
    start_memory = check_memory()
    test_chunk = pl.scan_parquet(file_path).limit(test_chunk_size).collect()
    memory_per_row = (check_memory() - start_memory) / test_chunk_size
    
    # Optimum chunk boyutunu hesapla
    optimal_chunk_size = int(max_memory_mb / memory_per_row) if memory_per_row > 0 else test_chunk_size
    print(f"Satır başına bellek kullanımı: {memory_per_row:.4f} MB")
    print(f"Optimum chunk boyutu: {optimal_chunk_size:,} satır")
    
    # Bu optimal boyutla işlem yap
    result = (
        pl.scan_parquet(file_path)
        .limit(optimal_chunk_size)
        .filter(pl.col("user_id_hashed").is_not_null())
        .group_by("user_id_hashed")
        .agg(pl.count().alias("session_count"))
        .collect()
    )
    
    final_memory = check_memory()
    print(f"Final bellek kullanımı: {final_memory - start_memory:.1f} MB")
    print(f"İşlenen satır sayısı: {len(result):,}")
    
    return result

memory_result = process_with_memory_limit(f"{DATA_PATH}/train_sessions.parquet")


## 3. Filtreleme Teknikleri

In [ ]:
# 3.1 Predicate Pushdown

print("=== PREDICATE PUSHDOWN ===")

print("\n--- Verimsiz Yöntem: Önce Yükle, Sonra Filtrele ---")
start_time = time.time()
start_memory = check_memory()

bad_approach = (
    pl.read_parquet(f"{DATA_PATH}/train_sessions.parquet")
    .filter(pl.col("user_id_hashed").is_not_null())
    .group_by("user_id_hashed")
    .agg(pl.count().alias("session_count"))
)

bad_time = time.time() - start_time
bad_memory = check_memory() - start_memory
print(f"Süre: {bad_time:.2f} saniye")
print(f"Bellek: {bad_memory:.1f} MB")
print(f"Sonuç: {len(bad_approach)} satır")

print("\n--- Optimize Yöntem: Lazy Evaluation ile Predicate Pushdown ---")
start_time = time.time()
start_memory = check_memory()

good_approach = (
    pl.scan_parquet(f"{DATA_PATH}/train_sessions.parquet")
    .filter(pl.col("user_id_hashed").is_not_null())  # Filtre dosya okuma sırasında uygulanır
    .group_by("user_id_hashed")
    .agg(pl.count().alias("session_count"))
    .collect()
)

good_time = time.time() - start_time
good_memory = check_memory() - start_memory
print(f"Süre: {good_time:.2f} saniye")
print(f"Bellek: {good_memory:.1f} MB")
print(f"Sonuç: {len(good_approach)} satır")

if good_time > 0:
    print(f"\n İyileştirme: {bad_time/good_time:.1f}x daha hızlı, {bad_memory/good_memory:.1f}x daha az bellek kullanımı.")


In [ ]:
# 3.2 Column Selection (Sütun Seçimi)

print("=== COLUMN SELECTION ===")

print("\n--- Tüm Kolonlar vs Seçili Kolonlar ---")

all_columns = pl.scan_parquet(f"{DATA_PATH}/train_sessions.parquet").collect_schema().names()
print(f"Dosyadaki tüm kolonlar: {all_columns}")

# Tüm kolonları yükle
start_time = time.time()
start_memory = check_memory()

all_cols_df = pl.scan_parquet(f"{DATA_PATH}/train_sessions.parquet").collect()

all_time = time.time() - start_time
all_memory = check_memory() - start_memory
print(f"\nTüm kolonlar - Süre: {all_time:.2f}s, Bellek: {all_memory:.1f} MB")

# Sadece gerekli kolonları yükle
start_time = time.time()
start_memory = check_memory()

selected_cols_df = (
    pl.scan_parquet(f"{DATA_PATH}/train_sessions.parquet")
    .select(["user_id_hashed", "session_id"])  # Sadece ihtiyaç duyulan kolonlar
    .collect()
)

selected_time = time.time() - start_time
selected_memory = check_memory() - start_memory
print(f"Seçili kolonlar - Süre: {selected_time:.2f}s, Bellek: {selected_memory:.1f} MB")

if selected_time > 0:
    print(f"\nİyileştirme: {all_time/selected_time:.1f}x daha hızlı")
if selected_memory > 0:
    print(f"Bellek tasarrufu: {all_memory/selected_memory:.1f}x daha az bellek")


Mevcut bellek kullanımı: 7636.9 MB

Tüm kolonlar - Süre: 0.13s, Bellek: 6.3 MB
Mevcut bellek kullanımı: 7636.9 MB
Mevcut bellek kullanımı: 7638.8 MB
Seçili kolonlar - Süre: 0.04s, Bellek: 1.9 MB

İyileştirme: 3.6x daha hızlı
Bellek tasarrufu: 3.3x daha az bellek


In [ ]:
# 3.3 Complex Filtering

print("=== COMPLEX FILTERING ===")

selected_file = f"{DATA_PATH}/content/search_log.parquet"
print(f"Kullanılan dosya: {selected_file}")

# Dosyanın metadatasını kontrol et
schema = pl.scan_parquet(selected_file).collect_schema()
print(f"Dosya kolonları: {list(schema.names())}")

# Ortak kolonları kullanarak filtreleme yap
complex_filter_query = (
    pl.scan_parquet(selected_file)
    .filter(pl.col("content_id_hashed").is_not_null())
    .group_by("content_id_hashed")
    .agg([
        pl.count().alias("record_count")
    ])
    .filter(pl.col("record_count") >= 2)  # En az 2 kayıt
    .sort("record_count", descending=True)
    .limit(100)
    )

print("\nKarmaşık filtreleme query planı:")
print(complex_filter_query.explain())

print("\nKarmaşık filtreleme çalıştırılıyor...")
start_time = time.time()
complex_result = complex_filter_query.collect()
end_time = time.time()

print(f"Süre: {end_time - start_time:.2f} saniye")
print(f"Sonuç: {len(complex_result)} kayıt bulundu")
print("\nİlk 5 sonuç:")
print(complex_result.head())


Süre: 2.07 saniye
Sonuç: 100 kayıt bulundu

İlk 5 sonuç:
shape: (5, 2)
┌───────────────────┬──────────────┐
│ content_id_hashed ┆ record_count │
│ ---               ┆ ---          │
│ str               ┆ u32          │
╞═══════════════════╪══════════════╡
│ 30aa610809113dc6  ┆ 35           │
│ 56bebb0006c13233  ┆ 35           │
│ 3df1b15b925fa10b  ┆ 35           │
│ 79dd63adcceaadce  ┆ 35           │
│ 06022f61d53000ea  ┆ 35           │
└───────────────────┴──────────────┘
